# Bilheteria

Estudo longitudinal das ações pagas em bilheteria.

## Acesso ao HIVE

In [19]:
import warnings
import numpy as np
import pandas as pd
import pyodbc
from pathlib import Path

OUTPUT_PATH = Path('output')
OUTPUT_PATH.mkdir(exist_ok=True)
DATA_INICIAL = '2015-01-01'

# Ajuste a string abaixo se o DSN exigir parÃ¢metros adicionais.
# Para autenticaÃ§Ã£o Kerberos, normalmente basta usar o DSN configurado.
KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Authentication=Kerberos;'
# Se o driver exigir Trusted Connection no Windows, use:
# KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Trusted_Connection=Yes;'


def get_connection() -> pyodbc.Connection:
    return pyodbc.connect(KERBEROS_CONN_STR, autocommit=True)


def query_to_df(query: str) -> pd.DataFrame:
    with get_connection() as conn:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore',
                message='.*SQLAlchemy connectable.*',
                category=UserWarning,
                module='pandas.io.sql',
            )
            return pd.read_sql_query(query, conn)


## Bilheteria

In [ ]:
sql_raw_bilheteria = f"""
select
    status_pedido             as status,
    nome_categoria_evento     as linguagem,
    nome_unidade              as unidade,
    espaco_nome               as local_uo,
    nome_evento               as evento,
    evento_id,
    sessao_id,
    data_sessao,
    hora_sessao,
    year(data_sessao)         as ano,
    month(data_sessao)        as mes,
    coalesce(sum(valor_ingresso_num), 0)                                     as receita_sessao,
    case when max(max_valor_evento) > 0 then 'Pago' else 'Gratuito' end     as tipo_evento,
    count(codigo_ingresso)                                                   as ingressos,
    count(distinct codigo_pedido)                                            as pedidos,
    count(distinct cod_transacao)                                            as transacoes
from (
    select
        status_pedido,
        nome_categoria_evento,
        nome_unidade,
        espaco_nome,
        nome_evento,
        codigo_ingresso,
        codigo_pedido,
        cod_transacao,
        coalesce(cast(replace(valor_ingresso, ',', '.') as double), 0)       as valor_ingresso_num,
        concat(cast(codigo_unidade as string), '_',
               cast(codigo_evento  as string))                               as evento_id,
        concat(cast(codigo_unidade as string), '_',
               cast(codigo_evento  as string), '_',
               cast(data_hora_sessao as string))                             as sessao_id,
        to_date(data_hora_sessao)                                            as data_sessao,
        date_format(cast(data_hora_sessao as timestamp), 'HH:mm')           as hora_sessao,
        max(coalesce(cast(replace(valor_ingresso, ',', '.') as double), 0))
            over (partition by codigo_unidade, codigo_evento)                as max_valor_evento
    from stg_bilheteria.vw_stg_bil_bilhetagem
    where nome_categoria_evento in (
        'CIRCO', 'CINEMA E VÍDEO', 'DANÇA', 'MUSICA', 'MÚSICA', 'TEATRO'
    )
) b
where status_pedido in ('Pagamento Confirmado', 'Bilhetes Validados')
group by
    status_pedido,
    nome_categoria_evento,
    nome_unidade,
    espaco_nome,
    nome_evento,
    evento_id,
    sessao_id,
    data_sessao,
    hora_sessao
"""

In [ ]:
df_raw_bilheteria = query_to_df(sql_raw_bilheteria)
display(df_raw_bilheteria)

Por usuário

In [24]:
sql_raw_bilheteria_users = f"""
with tipo_evento as (
    select
        codigo_unidade,
        codigo_evento,
        case
            when max(coalesce(cast(valor_ingresso as double), 0)) > 0 then 'Pago'
            else 'Gratuito'
        end as tipo_evento
    from stg_bilheteria.vw_stg_bil_bilhetagem
    group by
        codigo_unidade,
        codigo_evento
)
select
    coalesce(nullif(trim(b.cliente_credencial), ''), trim(b.cliente_cpf)) as id_cliente,
    count(case when b.status_pedido = 'Pagamento Confirmado' then b.codigo_ingresso end) as ingressos_confirmados,
    count(case when b.status_pedido = 'Bilhetes Validados' then b.codigo_ingresso end) as ingressos_validados,
    count(case when b.status_pedido = 'Pagamento Confirmado' then b.codigo_ingresso end)
      - count(case when b.status_pedido = 'Bilhetes Validados' then b.codigo_ingresso end) as nao_validados,
    case
        when count(case when b.status_pedido = 'Pagamento Confirmado' then b.codigo_ingresso end) = 0 then null
        else round(
            100.0 * count(case when b.status_pedido = 'Bilhetes Validados' then b.codigo_ingresso end)
            / count(case when b.status_pedido = 'Pagamento Confirmado' then b.codigo_ingresso end),
            2
        )
    end as perc_validados
from stg_bilheteria.vw_stg_bil_bilhetagem b
left join tipo_evento t
    on b.codigo_unidade = t.codigo_unidade
   and b.codigo_evento = t.codigo_evento
where 
    b.status_pedido in ('Pagamento Confirmado', 'Bilhetes Validados')
    and b.nome_categoria_evento in (
        'CIRCO',
        'CINEMA E VÍDEO',
        'DANÇA',
        'MUSICA',
        'MÚSICA',
        'TEATRO'
    )
    and t.tipo_evento = 'Pago'
group by
    coalesce(nullif(trim(b.cliente_credencial), ''), trim(b.cliente_cpf))
order by
	nao_validados desc, perc_validados asc, ingressos_confirmados desc
;
"""

In [25]:
df_raw_bilheteria_users = query_to_df(sql_raw_bilheteria_users)
display(df_raw_bilheteria_users)

C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_17192\1233636853.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


,id_cliente,ingressos_confirmados,ingressos_validados,nao_validados,perc_validados
0,5920322494,415,167,248,40.24
1,1520102393,566,339,227,59.89
2,8520311439,294,156,138,53.06
3,8620310246,163,38,125,23.31
4,5620301814,121,7,114,5.79
...,...,...,...,...,...
346445,2520408463,284,1159,-875,408.10
346446,8220317296,224,1214,-990,541.96
346447,820406331,273,1312,-1039,480.59
346448,320313555,419,2023,-1604,482.82


## Evolução de ingressos por linguagem
Pré-pandemia: 2016–2019 | Pós-pandemia: 2022–2025

In [ ]:

# ─── Configuração ─────────────────────────────────────────────────────────────
ANOS_PRE = [2018, 2019]
ANOS_POS = [2024, 2025]
ANOS_ALL = ANOS_PRE + ANOS_POS

NORM_LING = {
    'CIRCO':          'Circo',
    'CINEMA E VÍDEO': 'Cinema e Vídeo',
    'DANÇA':          'Dança',
    'MUSICA':         'Música',
    'MÚSICA':         'Música',
    'TEATRO':         'Teatro',
}

# ─── Preparo ──────────────────────────────────────────────────────────────────
df_bil = df_raw_bilheteria.copy()
df_bil['linguagem'] = df_bil['linguagem'].map(NORM_LING)
df_bil = df_bil[df_bil['ano'].isin(ANOS_ALL) & df_bil['linguagem'].notna()].copy()
df_bil['periodo'] = df_bil['ano'].apply(
    lambda a: 'Pré-pandemia' if a in ANOS_PRE else 'Pós-pandemia'
)

# ─── Agregações base ──────────────────────────────────────────────────────────
by_year = (
    df_bil.groupby(['linguagem', 'ano'])
    .agg(ingressos=('ingressos', 'sum'), sessoes=('sessao_id', 'nunique'))
    .reset_index()
)
by_periodo = (
    df_bil.groupby(['linguagem', 'periodo'])
    .agg(ingressos=('ingressos', 'sum'), sessoes=('sessao_id', 'nunique'))
    .reset_index()
)

# ─── Pivôs por ano ────────────────────────────────────────────────────────────
p_ing = by_year.pivot_table(
    index='linguagem', columns='ano', values='ingressos', fill_value=0
)
p_ses = by_year.pivot_table(
    index='linguagem', columns='ano', values='sessoes', fill_value=0
)
for _p in [p_ing, p_ses]:
    _p.columns.name = None
    _p.index.name = 'Linguagem'

pre_cols = [c for c in ANOS_PRE if c in p_ing.columns]
pos_cols = [c for c in ANOS_POS if c in p_ing.columns]

# ─── Helpers ──────────────────────────────────────────────────────────────────
def _var_pct(pos, pre):
    return round((pos - pre) / pre * 100, 1) if pre and pre != 0 else float('nan')

def _color_pct(v):
    if pd.isna(v):
        return ''
    return 'color: forestgreen; font-weight:bold' if v > 0 else 'color: crimson; font-weight:bold'


In [38]:

# ─── Tabela 1: Ingressos totais por linguagem × ano ───────────────────────────
t1 = p_ing[pre_cols + pos_cols].copy()
t1.index.name = 'Linguagem'

pre_ing = by_periodo[by_periodo['periodo'] == 'Pré-pandemia'].set_index('linguagem')['ingressos']
pos_ing = by_periodo[by_periodo['periodo'] == 'Pós-pandemia'].set_index('linguagem')['ingressos']

t1['Total Pré']       = pre_ing
t1['Total Pós']       = pos_ing
t1['Méd. Anual Pré']  = (t1['Total Pré'] / len(pre_cols)).round(0).astype(int)
t1['Méd. Anual Pós']  = (t1['Total Pós'] / len(pos_cols)).round(0).astype(int)
t1['Var. Total %']    = t1.apply(lambda r: _var_pct(r['Total Pós'],      r['Total Pré']),      axis=1)
t1['Var. Méd/Ano %']  = t1.apply(lambda r: _var_pct(r['Méd. Anual Pós'], r['Méd. Anual Pré']), axis=1)

# Linha TOTAL
_tp = t1['Total Pré'].sum(); _tpos = t1['Total Pós'].sum()
_mp = round(_tp / len(pre_cols));  _mpos = round(_tpos / len(pos_cols))
t1.loc['TOTAL'] = {
    **{c: t1[c].sum() for c in pre_cols + pos_cols},
    'Total Pré': _tp,       'Total Pós': _tpos,
    'Méd. Anual Pré': _mp,  'Méd. Anual Pós': _mpos,
    'Var. Total %':   _var_pct(_tpos, _tp),
    'Var. Méd/Ano %': _var_pct(_mpos, _mp),
}

col_t1     = pre_cols + ['Total Pré', 'Méd. Anual Pré'] + pos_cols + ['Total Pós', 'Méd. Anual Pós', 'Var. Total %', 'Var. Méd/Ano %']
fmt_int_t1 = {c: '{:,.0f}' for c in pre_cols + pos_cols + ['Total Pré', 'Total Pós', 'Méd. Anual Pré', 'Méd. Anual Pós']}
fmt_pct_t1 = {'Var. Total %': '{:+.1f}%', 'Var. Méd/Ano %': '{:+.1f}%'}

print("=== 1. INGRESSOS TOTAIS POR LINGUAGEM E ANO ===\n")
display(
    t1[col_t1].style
    .format({**fmt_int_t1, **fmt_pct_t1})
    .applymap(_color_pct, subset=list(fmt_pct_t1.keys()))
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
)

# ─── Tabela 2: Média de ingressos por sessão ──────────────────────────────────
# Usa razão de totais do período (não média de razões anuais)
t2 = (p_ing[pre_cols + pos_cols] / p_ses[pre_cols + pos_cols].replace(0, float('nan'))).round(1)
t2.index.name = 'Linguagem'

_pre_med = by_periodo[by_periodo['periodo'] == 'Pré-pandemia'].assign(
    m=lambda d: (d['ingressos'] / d['sessoes']).round(1)
).set_index('linguagem')['m'].rename('Média Pré')

_pos_med = by_periodo[by_periodo['periodo'] == 'Pós-pandemia'].assign(
    m=lambda d: (d['ingressos'] / d['sessoes']).round(1)
).set_index('linguagem')['m'].rename('Média Pós')

t2 = t2.join(_pre_med).join(_pos_med)
t2['Var. Méd/Sessão %'] = t2.apply(lambda r: _var_pct(r['Média Pós'], r['Média Pré']), axis=1)

col_t2    = pre_cols + ['Média Pré'] + pos_cols + ['Média Pós', 'Var. Méd/Sessão %']
fmt_flt   = {c: '{:.1f}' for c in pre_cols + pos_cols + ['Média Pré', 'Média Pós']}

print("\n=== 2. MÉDIA DE INGRESSOS POR SESSÃO ===\n")
display(
    t2[col_t2].style
    .format({**fmt_flt, 'Var. Méd/Sessão %': '{:+.1f}%'})
    .applymap(_color_pct, subset=['Var. Méd/Sessão %'])
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
)


=== 1. INGRESSOS TOTAIS POR LINGUAGEM E ANO ===



,2016,2017,2018,2019,Total Pré,Méd. Anual Pré,2022,2023,2024,2025,Total Pós,Méd. Anual Pós,Var. Total %,Var. Méd/Ano %
Linguagem,,,,,,,,,,,,,,
Cinema e Vídeo,"4,326","7,010","14,825","22,282","48,443","12,111","10,857","21,094","36,296","64,768","133,015","33,254",+174.6%,+174.6%
Circo,"1,883","5,472","3,107","5,871","16,333","4,083","7,420","16,490","15,596","20,248","59,754","14,938",+265.8%,+265.9%
Dança,"3,968","4,706","5,333","7,618","21,625","5,406","12,007","14,628","14,979","17,284","58,898","14,724",+172.4%,+172.4%
Música,"200,377","195,016","226,842","236,516","858,751","214,688","242,614","442,900","444,116","423,937","1,553,567","388,392",+80.9%,+80.9%
Teatro,"43,805","56,153","63,138","89,234","252,330","63,082","106,786","138,953","167,893","186,811","600,443","150,111",+138.0%,+138.0%
TOTAL,"254,359","268,357","313,245","361,521","1,197,482","299,370","379,684","634,065","678,880","713,048","2,405,677","601,419",+100.9%,+100.9%



=== 2. MÉDIA DE INGRESSOS POR SESSÃO ===



,2016,2017,2018,2019,Média Pré,2022,2023,2024,2025,Média Pós,Var. Méd/Sessão %
Linguagem,,,,,,,,,,,
Cinema e Vídeo,6.2,10.3,17.2,24.0,15.3,14.2,22.5,40.0,66.8,37.2,+143.1%
Circo,25.8,44.9,26.3,40.2,35.6,39.5,79.7,61.2,72.6,64.3,+80.6%
Dança,18.8,20.0,22.3,23.8,21.5,46.2,47.8,39.5,43.6,43.9,+104.2%
Música,129.5,128.6,132.7,140.8,133.1,151.6,231.6,228.5,234.0,213.8,+60.6%
Teatro,29.9,41.6,38.0,49.8,40.3,59.6,71.5,81.8,84.4,75.0,+86.1%


## Top 100 espaços: comparação entre biênios
Coorte fixada nos 100 espaços (`unidade + local_uo`) com maior volume de ingressos em 2018/2019.
Comparação com 2022/2023 e 2024/2025 elimina distorção de novos espaços abertos após a pandemia.

In [47]:

BIENIOS = {
    '2018/2019': [2018, 2019],
    '2022/2023': [2022, 2023],
    '2024/2025': [2024, 2025],
}
ANOS_BIE = [a for anos in BIENIOS.values() for a in anos]
CORTE_INGRESSOS = 10_000

# Chave composta: evita colapso de espaços homônimos em unidades diferentes
df_bie = df_raw_bilheteria[df_raw_bilheteria['ano'].isin(ANOS_BIE)].copy()
df_bie['bienio'] = df_bie['ano'].apply(
    lambda a: next(k for k, v in BIENIOS.items() if a in v)
)
df_bie['espaco'] = df_bie['unidade'].str.strip() + '  —  ' + df_bie['local_uo'].str.strip()

# ─── Coorte: espaços com mais de 10 mil ingressos no total dos 6 anos ─────────
total_por_espaco = df_bie.groupby('espaco')['ingressos'].sum().sort_values(ascending=False)
coorte = total_por_espaco[total_por_espaco > CORTE_INGRESSOS].index.tolist()

print(f"Espaços com mais de {CORTE_INGRESSOS:,} ingressos nos 6 anos: {len(coorte)}")
print(f"Total de espaços únicos no período: {total_por_espaco.shape[0]}")
print(f"\nDistribuição do total de ingressos por espaço:")
display(
    total_por_espaco.describe(percentiles=[.25, .5, .75, .9, .95])
    .rename('ingressos (6 anos)')
    .to_frame()
    .style.format('{:,.0f}')
)

df_top = df_bie[df_bie['espaco'].isin(coorte)].copy()

# ─── Agregação por espaço + biênio ───────────────────────────────────────────
by_bie = (
    df_top.groupby(['espaco', 'bienio'])
    .agg(ingressos=('ingressos', 'sum'), sessoes=('sessao_id', 'nunique'))
    .reset_index()
)
by_bie['media_sessao'] = (by_bie['ingressos'] / by_bie['sessoes']).round(1)

# Cobertura: quantos da coorte têm dados em cada biênio
cobertura = (
    by_bie[by_bie['ingressos'] > 0]
    .groupby('bienio')['espaco'].nunique()
    .reindex(BIENIOS.keys())
    .rename('espaços com ingressos > 0')
    .to_frame()
)
print(f"\nCobertura da coorte ({len(coorte)} espaços) por biênio:")
display(cobertura)


Espaços com mais de 10,000 ingressos nos 6 anos: 48
Total de espaços únicos no período: 256

Distribuição do total de ingressos por espaço:


,ingressos (6 anos)
count,256
mean,"12,033"
std,"36,844"
min,1
25%,110
50%,618
75%,"6,059"
90%,"27,036"
95%,"63,874"
max,"325,767"



Cobertura da coorte (48 espaços) por biênio:


,espaços com ingressos > 0
bienio,
2018/2019,43
2022/2023,46
2024/2025,48


In [48]:

# ─── Pivot: ingressos por espaço × biênio ────────────────────────────────────
p = by_bie.pivot_table(
    index='espaco', columns='bienio', values='ingressos', fill_value=0
)
p.columns.name = None
p.index.name = 'Espaço (Unidade — Local)'

for col in BIENIOS:
    if col not in p.columns:
        p[col] = 0

p['Total 6 anos']        = p['2018/2019'] + p['2022/2023'] + p['2024/2025']
p['Var 22/23 %']         = p.apply(lambda r: _var_pct(r['2022/2023'], r['2018/2019']), axis=1)
p['Var 24/25 %']         = p.apply(lambda r: _var_pct(r['2024/2025'], r['2018/2019']), axis=1)
p['Var 24/25 vs 22/23 %']= p.apply(lambda r: _var_pct(r['2024/2025'], r['2022/2023']), axis=1)

# Linha TOTAL (separada, fora da ordenação)
_t18 = p['2018/2019'].sum(); _t22 = p['2022/2023'].sum(); _t24 = p['2024/2025'].sum()
total_row = pd.DataFrame([{
    '2018/2019': _t18, '2022/2023': _t22, '2024/2025': _t24,
    'Total 6 anos':          _t18 + _t22 + _t24,
    'Var 22/23 %':           _var_pct(_t22, _t18),
    'Var 24/25 %':           _var_pct(_t24, _t18),
    'Var 24/25 vs 22/23 %':  _var_pct(_t24, _t22),
}], index=pd.Index([f'TOTAL ({len(coorte)} espaços)'], name='Espaço (Unidade — Local)'))

p_sorted = pd.concat([p.sort_values('Total 6 anos', ascending=False), total_row])

# ─── Pivot: média ingressos/sessão ───────────────────────────────────────────
p_m = by_bie.pivot_table(
    index='espaco', columns='bienio', values='media_sessao', fill_value=0
)
p_m.columns.name = None
p_m.index.name = 'Espaço (Unidade — Local)'
for col in BIENIOS:
    if col not in p_m.columns:
        p_m[col] = 0

p_m['Var 22/23 %'] = p_m.apply(lambda r: _var_pct(r['2022/2023'], r['2018/2019']), axis=1)
p_m['Var 24/25 %'] = p_m.apply(lambda r: _var_pct(r['2024/2025'], r['2018/2019']), axis=1)

_ordem = p.sort_values('Total 6 anos', ascending=False).index
p_m = p_m.reindex(_ordem)

# ─── Exibição ─────────────────────────────────────────────────────────────────
_bie_cols = ['2018/2019', '2022/2023', '2024/2025']
_pct_cols = ['Var 22/23 %', 'Var 24/25 %', 'Var 24/25 vs 22/23 %']
_all_cols  = _bie_cols + ['Total 6 anos'] + _pct_cols
_fmt_p = {**{c: '{:,.0f}' for c in _bie_cols + ['Total 6 anos']},
          **{c: '{:+.1f}%' for c in _pct_cols}}

print(f"=== {len(coorte)} ESPAÇOS COM > {CORTE_INGRESSOS:,} INGRESSOS — COMPARAÇÃO ENTRE BIÊNIOS ===\n")
display(
    p_sorted[_all_cols].style
    .format(_fmt_p, na_rep='-')
    .applymap(_color_pct, subset=_pct_cols)
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
)

_pct_cols_m = ['Var 22/23 %', 'Var 24/25 %']
_fmt_m = {**{c: '{:.1f}' for c in _bie_cols}, **{c: '{:+.1f}%' for c in _pct_cols_m}}

print(f"\n=== {len(coorte)} ESPAÇOS — MÉDIA DE INGRESSOS POR SESSÃO ===\n")
display(
    p_m[_bie_cols + _pct_cols_m].style
    .format(_fmt_m, na_rep='-')
    .applymap(_color_pct, subset=_pct_cols_m)
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
)


=== 48 ESPAÇOS COM > 10,000 INGRESSOS — COMPARAÇÃO ENTRE BIÊNIOS ===



,2018/2019,2022/2023,2024/2025,Total 6 anos,Var 22/23 %,Var 24/25 %,Var 24/25 vs 22/23 %
Espaço (Unidade — Local),,,,,,,
SESC PINHEIROS — TEATRO SESC PINHEIROS,"81,103","111,979","132,685","325,767",+38.1%,+63.6%,+18.5%
SESC POMPEIA — COMEDORIA POMPEIA,"72,816","93,018","110,954","276,788",+27.7%,+52.4%,+19.3%
SESC VILA MARIANA — TEATRO ANTUNES FILHO,"51,912","70,862","79,432","202,206",+36.5%,+53.0%,+12.1%
SESC BELENZINHO — COMEDORIA SESC BELENZINHO,"35,506","45,485","82,814","163,805",+28.1%,+133.2%,+82.1%
CINESESC — SALA CINESESC,"36,802","24,866","85,256","146,924",-32.4%,+131.7%,+242.9%
SESC POMPEIA — TEATRO SESC POMPEIA,"39,201","54,840","49,758","143,799",+39.9%,+26.9%,-9.3%
SESC CONSOLAÇÃO — TEATRO ANCHIETA,"24,851","40,536","53,734","119,121",+63.1%,+116.2%,+32.6%
SESC BOM RETIRO — TEATRO BOM RETIRO,"25,590","32,684","35,266","93,540",+27.7%,+37.8%,+7.9%
SESC BELENZINHO — TEATRO SESC BELENZINHO,"17,908","33,360","38,734","90,002",+86.3%,+116.3%,+16.1%



=== 48 ESPAÇOS — MÉDIA DE INGRESSOS POR SESSÃO ===



,2018/2019,2022/2023,2024/2025,Var 22/23 %,Var 24/25 %
Espaço (Unidade — Local),,,,,
SESC PINHEIROS — TEATRO SESC PINHEIROS,257.5,373.3,402.1,+45.0%,+56.2%
SESC POMPEIA — COMEDORIA POMPEIA,235.7,372.1,381.3,+57.9%,+61.8%
SESC VILA MARIANA — TEATRO ANTUNES FILHO,177.8,227.9,241.4,+28.2%,+35.8%
SESC BELENZINHO — COMEDORIA SESC BELENZINHO,182.1,269.1,426.9,+47.8%,+134.4%
CINESESC — SALA CINESESC,20.9,18.2,55.0,-12.9%,+163.2%
SESC POMPEIA — TEATRO SESC POMPEIA,149.6,232.4,235.8,+55.3%,+57.6%
SESC CONSOLAÇÃO — TEATRO ANCHIETA,78.4,93.4,110.3,+19.1%,+40.7%
SESC BOM RETIRO — TEATRO BOM RETIRO,76.8,86.5,99.1,+12.6%,+29.0%
SESC BELENZINHO — TEATRO SESC BELENZINHO,61.5,97.0,103.0,+57.7%,+67.5%


## Diagnóstico: divergência BI × notebook

O BI usa `vw_stg_bilheteria_web_balcao` (já agregada) enquanto este notebook usa `vw_stg_bil_bilhetagem` (um ingresso por linha).

Diferenças estruturais que podem causar a divergência:

| | Notebook | BI |
|---|---|---|
| View fonte | `vw_stg_bil_bilhetagem` | `vw_stg_bilheteria_web_balcao` |
| Filtro status | `Pagamento Confirmado` + `Bilhetes Validados` | **nenhum** |
| Filtro linguagem | 6 linguagens | **nenhum** |
| Métrica | `count(codigo_ingresso)` | `sum(qtde_ingressos_vendidos)` |
| Dimensão extra | — | `categoria_venda` (web / balcão) |

O objetivo aqui é isolar qual dessas diferenças explica o gap.

In [ ]:
# ─── Passo 1: reproduz exatamente a lógica do BI ─────────────────────────────
# Sem filtro de status, sem filtro de linguagem — igual ao Power Query
sql_bi_equivalente = """
select
    uo,
    codigo_evento                        as id_evento,
    nome_evento,
    data_sessao,
    hora_sessao,
    ano_sessao                           as ano,
    linguagem                            as categoria,
    info_status_venda                    as status_da_venda,
    categoria_venda,
    sum(qtde_bloqueio_tecnico)           as bloqueios,
    sum(qtde_ingressos_vendidos)         as vendidos,
    sum(qtde_ingressos_disponiveis)      as disponiveis,
    sum(valor_venda)                     as pagamentos,
    -- qtde_espaco: se > 1, pode haver duplicação de ingressos
    max(qtde_espaco)                     as max_qtde_espaco,
    count(1)                             as linhas_fonte
from stg_bilheteria.vw_stg_bilheteria_web_balcao
where ano_sessao >= 2018
group by
    uo,
    codigo_evento,
    nome_evento,
    data_sessao,
    hora_sessao,
    ano_sessao,
    linguagem,
    info_status_venda,
    categoria_venda
"""

df_bi = query_to_df(sql_bi_equivalente)
print(f"Linhas: {len(df_bi):,}")
print(f"Total vendidos (BI, sem filtro): {df_bi['vendidos'].sum():,.0f}")
display(df_bi.head())

In [ ]:
# ─── Passo 2: comparação por camadas de filtro ───────────────────────────────
LINGUAGENS_NOTEBOOK = {'CIRCO', 'CINEMA E VÍDEO', 'DANÇA', 'MUSICA', 'MÚSICA', 'TEATRO'}
STATUS_NOTEBOOK     = {'Pagamento Confirmado', 'Bilhetes Validados'}

total_notebook = df_raw_bilheteria['ingressos'].sum()

# Variantes do BI com filtros progressivos
bi_sem_filtro = df_bi['vendidos'].sum()

bi_status = df_bi[df_bi['status_da_venda'].isin(STATUS_NOTEBOOK)]['vendidos'].sum()

bi_status_ling = df_bi[
    df_bi['status_da_venda'].isin(STATUS_NOTEBOOK) &
    df_bi['categoria'].str.upper().isin(LINGUAGENS_NOTEBOOK)
]['vendidos'].sum()

# Hipótese categoria_venda: se web e balcão duplicam o mesmo ingresso,
# deduplica agrupando sem essa dimensão
bi_status_ling_dedup = (
    df_bi[
        df_bi['status_da_venda'].isin(STATUS_NOTEBOOK) &
        df_bi['categoria'].str.upper().isin(LINGUAGENS_NOTEBOOK)
    ]
    .groupby(['uo', 'id_evento', 'nome_evento', 'data_sessao', 'hora_sessao',
              'ano', 'categoria', 'status_da_venda'])['vendidos']
    .sum()
    .sum()
)

resumo = pd.DataFrame([
    {'cenário': 'Notebook (referência)',              'ingressos': total_notebook},
    {'cenário': 'BI — sem filtro',                    'ingressos': bi_sem_filtro},
    {'cenário': 'BI — filtrado por status',           'ingressos': bi_status},
    {'cenário': 'BI — filtrado por status + ling.',   'ingressos': bi_status_ling},
    {'cenário': 'BI — sem categoria_venda (dedup)',   'ingressos': bi_status_ling_dedup},
])
resumo['vs. notebook'] = resumo['ingressos'] - total_notebook
resumo['diferença %']  = ((resumo['ingressos'] / total_notebook - 1) * 100).round(1)

display(
    resumo.style
    .format({'ingressos': '{:,.0f}', 'vs. notebook': '{:+,.0f}', 'diferença %': '{:+.1f}%'})
    .set_properties(subset=['vs. notebook', 'diferença %'], **{'font-weight': 'bold'})
    .hide(axis='index')
)

In [ ]:
# ─── Passo 3: hipótese qtde_espaco — duplicação por espaço ───────────────────
# Se a view tem uma linha por espaço mas qtde_ingressos_vendidos é o total da sessão,
# o somatório está multiplicando ingressos pelo nº de espaços

suspeitos = df_bi[df_bi['max_qtde_espaco'] > 1]
print(f"Grupos com max_qtde_espaco > 1: {len(suspeitos):,} ({100*len(suspeitos)/len(df_bi):.1f}% do total)")
print(f"Vendidos nesses grupos: {suspeitos['vendidos'].sum():,.0f}")
print()

# Quantas linhas fonte somaram em cada grupo — se linhas_fonte > 1 e qtde_espaco > 1,
# pode haver inflação
print("Distribuição de linhas_fonte (qtd de linhas da view antes do GROUP BY):")
display(df_bi['linhas_fonte'].describe().to_frame().T.style.format('{:.1f}'))

# Mostra os status presentes na view — o notebook só aceita 2
print("\nStatus únicos na view (BI):")
display(
    df_bi.groupby('status_da_venda')['vendidos']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .style.format({'vendidos': '{:,.0f}'})
    .hide(axis='index')
)

# Distribuição de categorias/linguagens
print("\nCategorias únicas na view (BI):")
display(
    df_bi.groupby('categoria')['vendidos']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .style.format({'vendidos': '{:,.0f}'})
    .hide(axis='index')
)